# 2 - FWI gradient by the adjoint-state method + hockey-stick verification

Hand-code the adjoint-state gradient (forward solve + adjoint solve + correlation), confirm it equals the autodiff gradient to machine precision, and verify it with a Taylor / hockey-stick plot.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seidlr/acoustic-fwi-ndt-pytorch/blob/main/notebooks/02_adjoint_fwi_hockey_stick.ipynb)

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    OWNER = "seidlr"  # change to your account if you forked
    !git clone -q https://github.com/{OWNER}/acoustic-fwi-ndt-pytorch.git
    %cd acoustic-fwi-ndt-pytorch
    # Put the package on the path directly. An editable `pip install -e .` of a
    # src-layout package is not picked up by the already-running Colab kernel, and
    # torch/numpy/matplotlib are pre-installed on Colab, so no pip step is needed.
    sys.path.insert(0, os.path.abspath("src"))

import torch
from IPython.display import Image, display
from fwi.config import resolve_device, resolve_dtype

device = resolve_device()
dtype = resolve_dtype(device)
print("device:", device, "| dtype:", dtype)
print("Note: clean float64 needs CPU or CUDA; Apple MPS runs float32 (see caveat below).")

## Adjoint kernel vs autodiff
The manual adjoint correlates the adjoint field with the BARE forward Laplacian (not alpha2 * Laplacian) at cutoff=0, so it is the exact `dJ/d(alpha2)` - it must match `alpha2.grad`.

In [ ]:
from fwi.problems import build_problem
from fwi.adjoint import adjoint_gradient
from fwi.misfit import autodiff_gradient
from fwi import plotting

prob = build_problem('small', device=device, dtype=dtype)
kernel, J = adjoint_gradient(prob.start_alpha2, prob.observed, prob.src_sig,
    prob.src_i, prob.src_j, prob.rec_i, prob.rec_j, prob.cfg, active_mask=prob.active_mask)
g_ad, _ = autodiff_gradient(prob.start_alpha2, prob.observed, prob.src_sig,
    prob.src_i, prob.src_j, prob.rec_i, prob.rec_j, prob.cfg, active_mask=prob.active_mask)
rel = (kernel - g_ad).norm() / g_ad.norm()
print(f'adjoint vs autodiff relative-L2 = {float(rel):.2e}')
display(Image(str(plotting.save_field(kernel, 'outputs/nb2_kernel.png',
    title='adjoint-state kernel'))))

## Taylor / hockey-stick test
For step `h` along a random direction `dm`: the second-order remainder `r2 = |J(m+h dm) - J(m) - h<g,dm>|` must decay as `h^2` (slope 2) until round-off turns it back up - the hockey stick.

In [ ]:
from fwi.forward import forward
from fwi.misfit import l2_misfit
from fwi.gradient_test import taylor_test, clean_slope

def J_fn(m):
    with torch.no_grad():
        tr = forward(m, prob.src_sig, prob.src_i, prob.src_j, prob.rec_i, prob.rec_j, prob.cfg).traces
    return float(l2_misfit(tr, prob.observed, prob.cfg.dt))

torch.manual_seed(0)
dm = torch.randn_like(kernel) * prob.active_mask.to(kernel.dtype)
hs = [10.0**k for k in range(6, -7, -1)]
hs, r1, r2 = taylor_test(J_fn, prob.start_alpha2, dm, kernel, hs)
slope2, n = clean_slope(hs, r2)
print(f'second-order slope = {slope2:.3f} over {n} clean steps (expect ~2.0)')
display(Image(str(plotting.save_hockey_stick(hs, r1, r2, 'outputs/nb2_hockey.png',
    title=f'hockey-stick (slope2={slope2:.2f}, {dtype})'))))

### MPS (Apple GPU) caveat
Apple MPS has no float64, so it runs float32. The Taylor test then hits its round-off floor at a much larger step `h` (~1e-3 instead of ~1e-15), so the clean `h^2` range is shorter and the stick turns up earlier. The slope is still ~2 over the (shorter) clean range. For the cleanest hockey stick, run on CPU or CUDA (both float64). The source amplitude is also auto-lifted on float32 to avoid underflow - the physics and recovered model are unchanged.

## Adjoint-driven inversion

In [ ]:
from fwi.inversion import invert

alpha2_hat, history = invert(prob.start_alpha2, prob.observed, prob.src_sig,
    prob.src_i, prob.src_j, prob.rec_i, prob.rec_j, prob.cfg,
    active_mask=prob.active_mask, grad_fn='adjoint', n_iter=60)
print(f'misfit {history[0]:.3e} -> {history[-1]:.3e} ({history[0]/history[-1]:.1f}x)')
display(Image(str(plotting.save_convergence(history, 'outputs/nb2_conv.png'))))